# Summary

This notebook cleans the two raw and dirty chemical datasets,
`1991Che_DRSites.xls` and `log21P_MarjorChe_Sites.xlsx` stored in the `two_raw_datasets` folder. 

They contain the whole available chemical data, but are not in consistent formats and having some hidden issues within them.

The cleaning process is listed with these targeted issues in the following steps:

### Clean the log21 major chemical dataset

**Input**: the originally raw and dirty `log21P_MarjorChe_Sites.xlsx` dataset.

**Process**: 

Remove the rows with all missings and Retain needed columns $\rightarrow$ 

Back-transform the log2(Y+1) to the original Y values $\rightarrow$

Infer the units of all 16 chemicals $\rightarrow$ 

Internally uniform the units of Hg and Zn values $\rightarrow$

Record units.

**Output**: a cleaned chemical data set of shape (245, 22), ready for merging.

**Details**:

Specifically using the third sheet in the `log21P_MarjorChe_Sites.xlsx` file, which contains the chemical data mainly collected in 1999, 2004 and 2005.

1. Remove the rows with all missings in all chemical columns, which are the rows with year label - 1991.
Retain only the needed 16 chemical columns with site IDs and relevant information.

1. Back-transform the log2(Y+1) values to the original Y values.
   
2. Infer the units of these chemicals in Y values.
    
    Reasons:
   - There were no attached units in the original dataset.
   - These chemicals were apparently in different units as some chemicals are trivial elements needing smaller scales, while some chemicals are compounds needing larger scales.

3. Inspect the **Hg** and **Zn** columns and internally uniform the units of values within them. 
   
   That is: the 2004/2005 Hg values in **ug/g** are $\rightarrow$ **ng/g**; the 1999 Zn values in **mg/g** are $\rightarrow$ **ug/g**.

   $$\text{Hg}_{ug/g} \times 1000 = \text{Hg}_{ng/g}$$
    $$\text{Zn}_{mg/g} \times 1000 = \text{Zn}_{ug/g}$$
   
   Reasons:
   - The Hg values collected in 2004 and 2005 were in **ug/g** unit, while the rest of the Hg values were in **ng/g** unit. These 2004 and 2005 values are apparently **smaller** than others and perfectly related with the year labels.
 
   - The Zn values collected in 2004 and 2005 were in **ug/g** unit, while the rest values were in **mg/g**. These 2004 and 2005 values are much larger than others and perfectly related with the year labels.

**Conclusion**: After the steps from 0 to 3, the original ``log21P_MarjorChe_Sites.xlsx`` dataset is cleaned with 245 rows of complete chemical data in internally uniformed units. Not all the chemicals are in same units, some are in ug/g and other are in ng/g, this needs to be operated when merging with the `1991Che_DRSites.xlx` dataset.

### Clean the 1991 chemical dataset

**Input**: the originally raw and dirty `1991Che_DRSites.xls` dataset.

**Process**: Retain needed columns $\rightarrow$ Fill missings $\rightarrow$ Record units.

**Output**: the cleaned 1991 chemical data of shape (77, 22) ready for merging.

**Details**:

0. Retain the same 16 targeted chemical columns with site IDs and relevant information.

1. Fill the missing values in Hg(1), SumPCBs(45), Cd(2), OCS(63) and p,p'-DDE(52).
Apply the same filling approach with unifom random values draw from interval [0.01X, 0.5X], where X is the detection limit of each chemical. The detection limits X recorded in the `1991Che_DRSites.xls` data were in consistent units as their reponsible chemicals, not uniformly but internally consistent.

    Reasons:
    - There were missing values in this dataset, specifically 1 missing in Hg, 45 missings in SumPCBs, 2 missings in Cd, 63 missings in OCS and 52 missings in p,p'-DDE. 
    - The filling approach is based on Jian's procedure in 2008, page 31.

2. Record the units of these chemicals in the cleaned 1991 DR dataset.
   
   Reasons:
      - The unit array of the 16 chemicals in this 1991 dataset is not same as the unit array of the 16 chemicals in the cleaned log21 major chemical dataset.
      - In order to merge these two cleaned datasets, their units need to be compared first and then transform them into a uniformed unit array.

**Conclusions**: As the units scales were already clean in the original 1991 dataset, not much inspection work was needed. Retaining the necessary chemical columns and filling the missings makes the 1991 dataset clean and ready for merging. The last and must thing is to record the units of these chemicals in the cleaned 1991 dataset, which will and must guide how to merge it with the cleaned log21 major chemical dataset.

### Merge the two cleaned datasets for a clean and complete chemical dataset

# Clean log21 major chemical data

This section cleans the `log21P_MajorChe_Sites.xlsx` workbook, keeps the target metadata and 16 major-chemistry columns, back-transforms the `log2(X + 1)` chemistry values, infers the working units, and writes the cleaned workbook to `intermiediate_datasets`.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path.cwd()
INPUT_PATH = DATA_DIR / 'log21P_MajorChe_Sites.xlsx'
OUTPUT_DIR = DATA_DIR.parent / 'intermiediate_datasets'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'cleaned_and_inferred_major_chemical_data.xlsx'

META_SOURCE_COLS = {
    'Integrated Code': 'Integrated Code',
    'Station No ID': 'Station No ID  (Zhang)',
    'waterbody': 'Waterbody',
    'Year sampled': 'Year sampled',
    'Latitude': 'Latitude',
    'Longitude': 'Longitude',
}
CHEMICAL_SOURCE_COLS = {
    'Co': 'LCo',
    'Al': 'LAl',
    'Ni': 'LNi',
    'Mn': 'LMn',
    'Fe': 'LFe',
    'Cr': 'LCr',
    'Cu': 'LCu',
    'Hg': 'LHg',
    'Pb': 'LPb',
    'Zn': 'LZn',
    'SumPCBs': 'LSPCB',
    'Cd': 'LCd',
    'OCS': 'LOCS',
    "p,p'-DDE": 'LDDE',
    'As': 'LAs',
    'Ca': 'LCa',
}
CHEMICAL_ORDER = list(CHEMICAL_SOURCE_COLS)
NGG_CHEMICALS = {'Hg', 'SumPCBs', 'OCS', "p,p'-DDE"}
RENAME_MAP = {
    **{source: cleaned for cleaned, source in META_SOURCE_COLS.items()},
    **{source: cleaned for cleaned, source in CHEMICAL_SOURCE_COLS.items()},
}

source_df = pd.read_excel(INPUT_PATH)

missing_meta = [source for source in META_SOURCE_COLS.values() if source not in source_df.columns]
missing_chem = [source for source in CHEMICAL_SOURCE_COLS.values() if source not in source_df.columns]
if missing_meta or missing_chem:
    raise KeyError(
        'Missing expected source columns: '
        + ', '.join(missing_meta + missing_chem)
    )

cleaned_df = source_df[[*META_SOURCE_COLS.values(), *CHEMICAL_SOURCE_COLS.values()]].copy()
cleaned_df = cleaned_df.rename(columns=RENAME_MAP)

for col in ['Integrated Code', 'Station No ID', 'waterbody']:
    cleaned_df[col] = cleaned_df[col].astype('string').str.strip()
    cleaned_df.loc[cleaned_df[col] == '', col] = pd.NA

cleaned_df['Year sampled'] = pd.to_numeric(cleaned_df['Year sampled'], errors='coerce').astype('Int64')
cleaned_df['Latitude'] = pd.to_numeric(cleaned_df['Latitude'], errors='coerce')
cleaned_df['Longitude'] = pd.to_numeric(cleaned_df['Longitude'], errors='coerce')

for chem in CHEMICAL_ORDER:
    log_values = pd.to_numeric(cleaned_df[chem], errors='coerce')
    cleaned_df[chem] = np.power(2.0, log_values) - 1.0

# Keep rows that correspond to identifiable sites and contain at least one target chemical value.
cleaned_df = cleaned_df.loc[
    cleaned_df['Integrated Code'].notna()
    & cleaned_df[CHEMICAL_ORDER].notna().any(axis=1)
].reset_index(drop=True)

cleaned_df = cleaned_df[
    ['Integrated Code', 'Station No ID', 'waterbody', 'Year sampled', 'Latitude', 'Longitude', *CHEMICAL_ORDER]
]

unit_rows = []
for chem in CHEMICAL_ORDER:
    values = cleaned_df[chem].dropna()
    min_value = float(values.min()) if not values.empty else np.nan
    median_value = float(values.median()) if not values.empty else np.nan
    q75_value = float(values.quantile(0.75)) if not values.empty else np.nan
    max_value = float(values.max()) if not values.empty else np.nan

    if chem in NGG_CHEMICALS:
        inferred_unit = 'ng/g'
        reasoning = (
            f"Back-transformed distribution for {chem} (median={median_value:.4g}, q75={q75_value:.4g}, max={max_value:.4g}) "
            'fits contaminant-scale reporting best as ng/g.'
        )
    else:
        inferred_unit = 'ug/g'
        if chem in {'Al', 'Fe', 'Ca'}:
            reasoning = (
                f"Back-transformed distribution for {chem} (median={median_value:.4g}, q75={q75_value:.4g}, max={max_value:.4g}) "
                'fits major-element reporting in ug/g for this log21 file; mg/g would be unrealistically large.'
            )
        else:
            reasoning = (
                f"Back-transformed distribution for {chem} (median={median_value:.4g}, q75={q75_value:.4g}, max={max_value:.4g}) "
                'fits trace-metal reporting best as ug/g.'
            )

    unit_rows.append({
        'Chemical': chem,
        'Original log2(X + 1) column': CHEMICAL_SOURCE_COLS[chem],
        'Inferred unit': inferred_unit,
        'Min converted value': min_value,
        'Median converted value': median_value,
        '75th percentile converted value': q75_value,
        'Max converted value': max_value,
        'Inference notes': reasoning,
    })

units_df = pd.DataFrame(unit_rows)

with pd.ExcelWriter(OUTPUT_PATH) as writer:
    cleaned_df.to_excel(writer, sheet_name='cleaned_data', index=False)
    units_df.to_excel(writer, sheet_name='inferred_units', index=False)

print(f'Source rows read: {len(source_df)}')
print(f'Cleaned rows written: {len(cleaned_df)}')
print(f'Output workbook: {OUTPUT_PATH}')
print('\nCleaned data preview:')
display(cleaned_df.head())
print('\nInferred units:')
display(units_df)


Source rows read: 326
Cleaned rows written: 245
Output workbook: /Users/gufeng/2025_Winter/Thesis_Project/ThesisProject/CodeBase/MetaData/FinalCleaningWork/intermiediate_datasets/cleaned_and_inferred_major_chemical_data.xlsx

Cleaned data preview:


,Integrated Code,Station No ID,waterbody,Year sampled,Latitude,Longitude,Co,Al,Ni,Mn,...,Cu,Hg,Pb,Zn,SumPCBs,Cd,OCS,"p,p'-DDE",As,Ca
0,DR-01,001A,DR,1999,NaN,NaN,6.397981,9232.496801,21.233370,268.271425,...,18.021903,2139.572859,29.748263,0.039422,0.489231,1.725842,0.212312,0.214792,29.559960,47007.347786
1,DR-02,003ABC,DR,1999,42.353685,-82.944344,3.653624,4742.524828,4.307142,257.010460,...,11.123978,983.426986,22.630486,0.018371,3.973114,1.270737,0.020000,0.570591,23.452117,54338.514812
2,DR-03,004ABC,DR,1999,42.350308,-82.934247,3.164832,3509.922179,3.394234,33.163576,...,8.718024,648.958161,0.000400,0.057245,0.814339,0.224713,0.010000,0.182532,0.832938,48502.333729
3,DR-04,005ABC,DR,1999,42.342434,-82.945587,2.984709,3670.729827,2.253268,112.384890,...,4.487565,580.952357,13.744258,0.023481,2.205706,0.673739,0.024000,0.391947,12.978875,16074.532338
4,DR-05,<NA>,DR,<NA>,NaN,NaN,4.150037,4355.282463,4.571083,128.757235,...,6.770378,750.796852,19.121414,0.031831,5.257706,1.010720,0.507738,0.383416,20.025567,27584.799492



Inferred units:


,Chemical,Original log2(X + 1) column,Inferred unit,Min converted value,Median converted value,75th percentile converted value,Max converted value,Inference notes
0,Co,LCo,ug/g,1.455000e+00,5.084000,7.099143,14.488175,Back-transformed distribution for Co (median=5...
1,Al,LAl,ug/g,1.397000e+03,5014.000000,9296.197084,29257.989285,Back-transformed distribution for Al (median=5...
2,Ni,LNi,ug/g,2.097337e+00,11.110000,19.113543,686.120673,Back-transformed distribution for Ni (median=1...
3,Mn,LMn,ug/g,3.316358e+01,160.097334,222.634909,491.345448,Back-transformed distribution for Mn (median=1...
4,Fe,LFe,ug/g,1.700000e-04,13032.138839,20537.689781,61685.947075,Back-transformed distribution for Fe (median=1...
5,Cr,LCr,ug/g,2.220000e+00,13.840000,25.960000,121.652934,Back-transformed distribution for Cr (median=1...
6,Cu,LCu,ug/g,3.000000e-04,18.100000,34.550530,216.400000,Back-transformed distribution for Cu (median=1...
7,Hg,LHg,ng/g,1.494000e-02,526.862035,1636.787451,6939.498156,Back-transformed distribution for Hg (median=5...
8,Pb,LPb,ug/g,1.200000e-04,3.334000,11.412440,168.200000,Back-transformed distribution for Pb (median=3...
9,Zn,LZn,ug/g,4.565510e-07,0.543009,31.170000,332.000000,Back-transformed distribution for Zn (median=0...


# Fixing one: Hg data in 2004 and 2005 were not in ng/g but in ug/g

In [3]:
from pathlib import Path

import pandas as pd

major_output_path = globals().get('OUTPUT_PATH')
if major_output_path is None:
    major_output_path = Path.cwd().parent / 'intermiediate_datasets' / 'cleaned_and_inferred_major_chemical_data.xlsx'
major_output_path = Path(major_output_path)

corrected_major_df = pd.read_excel(major_output_path, sheet_name='cleaned_data')
corrected_units_df = pd.read_excel(major_output_path, sheet_name='inferred_units')

for col in ['Integrated Code', 'Station No ID', 'waterbody']:
    corrected_major_df[col] = corrected_major_df[col].astype('string').str.strip()
    corrected_major_df.loc[corrected_major_df[col] == '', col] = pd.NA

corrected_major_df['Year sampled'] = pd.to_numeric(corrected_major_df['Year sampled'], errors='coerce').astype('Int64')
corrected_major_df['Latitude'] = pd.to_numeric(corrected_major_df['Latitude'], errors='coerce')
corrected_major_df['Longitude'] = pd.to_numeric(corrected_major_df['Longitude'], errors='coerce')
corrected_major_df['Hg'] = pd.to_numeric(corrected_major_df['Hg'], errors='coerce')

hg_correction_mask = corrected_major_df['Hg'].notna() & (corrected_major_df['Year sampled'] > 1999)
hg_before_df = corrected_major_df.loc[
    hg_correction_mask,
    ['Integrated Code', 'Station No ID', 'Year sampled', 'Hg'],
].copy()

corrected_major_df.loc[hg_correction_mask, 'Hg'] = corrected_major_df.loc[hg_correction_mask, 'Hg'] * 1000.0

hg_after_df = corrected_major_df.loc[
    hg_correction_mask,
    ['Integrated Code', 'Station No ID', 'Year sampled', 'Hg'],
].copy()

hg_preview_df = hg_before_df.head(10).rename(columns={'Hg': 'Hg_before'})
hg_preview_df['Hg_after'] = hg_after_df.head(10)['Hg'].to_numpy()

hg_values = corrected_major_df['Hg'].dropna()
hg_unit_mask = corrected_units_df['Chemical'].eq('Hg')
if not hg_unit_mask.any():
    raise KeyError("Could not find the 'Hg' row in the inferred_units sheet.")

corrected_units_df.loc[hg_unit_mask, 'Inferred unit'] = 'ng/g'
corrected_units_df.loc[hg_unit_mask, 'Min converted value'] = float(hg_values.min()) if not hg_values.empty else pd.NA
corrected_units_df.loc[hg_unit_mask, 'Median converted value'] = float(hg_values.median()) if not hg_values.empty else pd.NA
corrected_units_df.loc[hg_unit_mask, '75th percentile converted value'] = float(hg_values.quantile(0.75)) if not hg_values.empty else pd.NA
corrected_units_df.loc[hg_unit_mask, 'Max converted value'] = float(hg_values.max()) if not hg_values.empty else pd.NA
corrected_units_df.loc[hg_unit_mask, 'Inference notes'] = (
    'Mercury remains reported in ng/g. Rows with Year sampled > 1999 were multiplied by 1000 because '
    'the 2004 and 2005 Hg values were recorded in ug/g in the source workbook and must be harmonized to ng/g.'
)

with pd.ExcelWriter(major_output_path) as writer:
    corrected_major_df.to_excel(writer, sheet_name='cleaned_data', index=False)
    corrected_units_df.to_excel(writer, sheet_name='inferred_units', index=False)

cleaned_df = corrected_major_df
units_df = corrected_units_df
if 'major_data_final' in globals():
    major_data_final = corrected_major_df.copy()
if 'major_units_final' in globals():
    major_units_final = corrected_units_df.copy()

print(f'Hg rows corrected for years > 1999: {int(hg_correction_mask.sum())}')
print(f'Rewritten cleaned major workbook: {major_output_path}')
print('\nHg correction preview:')
display(hg_preview_df)
print('\nUpdated Hg inferred-unit row:')
display(corrected_units_df.loc[hg_unit_mask].reset_index(drop=True))

Hg rows corrected for years > 1999: 105
Rewritten cleaned major workbook: /Users/gufeng/2025_Winter/Thesis_Project/ThesisProject/CodeBase/MetaData/FinalCleaningWork/intermiediate_datasets/cleaned_and_inferred_major_chemical_data.xlsx

Hg correction preview:


,Integrated Code,Station No ID,Year sampled,Hg_before,Hg_after
16,DR-140,S100,2004,0.17450,174.50
17,DR-141,S101,2004,0.03865,38.65
19,DR-143,S102,2004,0.03971,39.71
20,DR-144,S81,2004,0.02038,20.38
21,DR-145,S82,2004,0.04677,46.77
22,DR-146,S83,2004,0.22870,228.70
23,DR-147,S84,2004,0.29850,298.50
24,DR-148,S85,2004,6.29700,6297.00
25,DR-149,S87,2004,0.09966,99.66
27,DR-150,S89,2004,0.51640,516.40



Updated Hg inferred-unit row:


,Chemical,Original log2(X + 1) column,Inferred unit,Min converted value,Median converted value,75th percentile converted value,Max converted value,Inference notes
0,Hg,LHg,ng/g,14.94,648.766998,1734.036209,6939.498156,Mercury remains reported in ng/g. Rows with Ye...


# Fixing two: Zn data in 1999 were not in ug/g but in mg/g

In [4]:
# To do: take out the Zn values with years of 1999, and exceptionally for the group - 
# (DR-05, DR-14, DR-162, DR-166, DR-172, DR-173, DR-174, DR-179, DR-185, DR-236), which missed their years of 1999
# but they are. These sites having their Zn values in mg/g unit, and it needs to time with 1000 to make them
# in ug/g unit.

In [5]:
from pathlib import Path

import pandas as pd

major_output_path = globals().get('OUTPUT_PATH')
if major_output_path is None:
    major_output_path = Path.cwd().parent / 'intermiediate_datasets' / 'cleaned_and_inferred_major_chemical_data.xlsx'
major_output_path = Path(major_output_path)

corrected_major_df = pd.read_excel(major_output_path, sheet_name='cleaned_data')
corrected_units_df = pd.read_excel(major_output_path, sheet_name='inferred_units')

for col in ['Integrated Code', 'Station No ID', 'waterbody']:
    corrected_major_df[col] = corrected_major_df[col].astype('string').str.strip()
    corrected_major_df.loc[corrected_major_df[col] == '', col] = pd.NA

corrected_major_df['Year sampled'] = pd.to_numeric(corrected_major_df['Year sampled'], errors='coerce').astype('Int64')
corrected_major_df['Latitude'] = pd.to_numeric(corrected_major_df['Latitude'], errors='coerce')
corrected_major_df['Longitude'] = pd.to_numeric(corrected_major_df['Longitude'], errors='coerce')
corrected_major_df['Zn'] = pd.to_numeric(corrected_major_df['Zn'], errors='coerce')

exception_integrated_codes = {
    'DR-05', 'DR-14', 'DR-162', 'DR-166', 'DR-172',
    'DR-173', 'DR-174', 'DR-179', 'DR-185', 'DR-192', 'DR-236',
    
}
integrated_code_norm = corrected_major_df['Integrated Code'].astype('string').str.strip().str.upper()
year_numeric = pd.to_numeric(corrected_major_df['Year sampled'], errors='coerce')
year_is_1999 = year_numeric.eq(1999)
if hasattr(year_is_1999, 'fillna'):
    year_is_1999 = year_is_1999.fillna(False)
zn_1999_mask = corrected_major_df['Zn'].notna() & year_is_1999
zn_exception_mask = corrected_major_df['Zn'].notna() & integrated_code_norm.isin(exception_integrated_codes)
zn_correction_mask = zn_1999_mask | zn_exception_mask
zn_exception_only_mask = zn_exception_mask & ~year_is_1999

zn_before_df = corrected_major_df.loc[
    zn_correction_mask,
    ['Integrated Code', 'Station No ID', 'Year sampled', 'Zn'],
].copy()

zn_unit_mask = corrected_units_df['Chemical'].eq('Zn')
if not zn_unit_mask.any():
    raise KeyError("Could not find the 'Zn' row in the inferred_units sheet.")

zn_correction_note = (
    'Zinc remains reported in ug/g. Rows with Year sampled == 1999 and the exception sites '
    'DR-05, DR-14, DR-162, DR-166, DR-172, DR-173, DR-174, DR-179, DR-185, and DR-236 '
    'were recorded in mg/g in the source workbook and were multiplied by 1000 to harmonize to ug/g.'
)
existing_zn_note = str(corrected_units_df.loc[zn_unit_mask, 'Inference notes'].iloc[0])
zn_conversion_applied = zn_correction_note not in existing_zn_note

if zn_conversion_applied:
    corrected_major_df.loc[zn_correction_mask, 'Zn'] = corrected_major_df.loc[zn_correction_mask, 'Zn'] * 1000.0

zn_after_df = corrected_major_df.loc[
    zn_correction_mask,
    ['Integrated Code', 'Station No ID', 'Year sampled', 'Zn'],
].copy()

zn_preview_df = zn_before_df.rename(columns={'Zn': 'Zn_before'})
zn_preview_df['Zn_after'] = zn_after_df['Zn'].to_numpy()

zn_values = corrected_major_df['Zn'].dropna()
corrected_units_df.loc[zn_unit_mask, 'Inferred unit'] = 'ug/g'
corrected_units_df.loc[zn_unit_mask, 'Min converted value'] = float(zn_values.min()) if not zn_values.empty else pd.NA
corrected_units_df.loc[zn_unit_mask, 'Median converted value'] = float(zn_values.median()) if not zn_values.empty else pd.NA
corrected_units_df.loc[zn_unit_mask, '75th percentile converted value'] = float(zn_values.quantile(0.75)) if not zn_values.empty else pd.NA
corrected_units_df.loc[zn_unit_mask, 'Max converted value'] = float(zn_values.max()) if not zn_values.empty else pd.NA
corrected_units_df.loc[zn_unit_mask, 'Inference notes'] = zn_correction_note

with pd.ExcelWriter(major_output_path) as writer:
    corrected_major_df.to_excel(writer, sheet_name='cleaned_data', index=False)
    corrected_units_df.to_excel(writer, sheet_name='inferred_units', index=False)

cleaned_df = corrected_major_df
units_df = corrected_units_df
if 'major_data_final' in globals():
    major_data_final = corrected_major_df.copy()
if 'major_units_final' in globals():
    major_units_final = corrected_units_df.copy()

print(f'Zn rows matched by Year sampled == 1999: {int(zn_1999_mask.sum())}')
print(f'Zn exception-only rows corrected without a 1999 year value: {int(zn_exception_only_mask.sum())}')
print(f'Total Zn rows targeted for correction: {int(zn_correction_mask.sum())}')
print(f'Zn conversion applied in this run: {zn_conversion_applied}')
print(f'Rewritten cleaned major workbook: {major_output_path}')
print('\nZn correction preview:')
display(zn_preview_df)
print('\nUpdated Zn inferred-unit row:')
display(corrected_units_df.loc[zn_unit_mask].reset_index(drop=True))

Zn rows matched by Year sampled == 1999: 129
Zn exception-only rows corrected without a 1999 year value: 11
Total Zn rows targeted for correction: 140
Zn conversion applied in this run: True
Rewritten cleaned major workbook: /Users/gufeng/2025_Winter/Thesis_Project/ThesisProject/CodeBase/MetaData/FinalCleaningWork/intermiediate_datasets/cleaned_and_inferred_major_chemical_data.xlsx

Zn correction preview:


,Integrated Code,Station No ID,Year sampled,Zn_before,Zn_after
0,DR-01,001A,1999,0.039422,39.422444
1,DR-02,003ABC,1999,0.018371,18.370618
2,DR-03,004ABC,1999,0.057245,57.245478
3,DR-04,005ABC,1999,0.023481,23.481343
4,DR-05,<NA>,<NA>,0.031831,31.830877
...,...,...,...,...,...
153,DR-59,150B,1999,1.766082,1766.082159
154,DR-65,016C,1999,0.064820,64.820018
155,DR-76,017B,1999,0.102092,102.092481
156,DR-87,018A,1999,0.022457,22.457479



Updated Zn inferred-unit row:


,Chemical,Original log2(X + 1) column,Inferred unit,Min converted value,Median converted value,75th percentile converted value,Max converted value,Inference notes
0,Zn,LZn,ug/g,0.000457,57.372967,248.46202,2031.402966,Zinc remains reported in ug/g. Rows with Year ...


In [12]:
print(f"The shape of the corrected major dataframe is {corrected_major_df.shape}")

The shape of the corrected major dataframe is (245, 22)


## Clean 1991 DR chemical data

This section cleans the `1991Che_DRSites.xls` workbook using the `Tranposed` sheet, keeps the same metadata fields and 16 target chemicals, fills missing chemical values from the detection-limit row, and writes the cleaned 1991 workbook to `intermiediate_datasets`.

In [7]:
DR_INPUT_PATH = DATA_DIR / '1991Che_DRSites.xls'
DR_OUTPUT_PATH = OUTPUT_DIR / 'cleaned_1991_DR_sites.xlsx'

DR_SHEET_NAME = 'Tranposed'
CHEMICAL_ORDER_1991 = [
    'Co', 'Al', 'Ni', 'Mn', 'Fe', 'Cr', 'Cu', 'Hg',
    'Pb', 'Zn', 'SumPCBs', 'Cd', 'OCS', "p,p'-DDE", 'As', 'Ca',
]


def _norm_key(value):
    return ''.join(ch for ch in str(value).upper() if ch.isalnum())


def _find_column(columns, *candidates):
    norm_map = {_norm_key(col): col for col in columns}
    for candidate in candidates:
        key = _norm_key(candidate)
        if key in norm_map:
            return norm_map[key]
    raise KeyError(f'Could not resolve any of {candidates!r} from the 1991 transposed sheet columns.')


source_1991_df = pd.read_excel(DR_INPUT_PATH, sheet_name=DR_SHEET_NAME, header=0)
first_col_1991 = source_1991_df.columns[0]

units_row_1991 = source_1991_df.loc[
    source_1991_df[first_col_1991].astype('string').str.strip().eq('Units')
].iloc[0]
detection_row_1991 = source_1991_df.loc[
    source_1991_df[first_col_1991].astype('string').str.strip().eq('Detection Limit')
].iloc[0]

meta_cols_1991 = {
    'Integrated Code': _find_column(source_1991_df.columns, 'Integrated Code'),
    'Station No ID': _find_column(source_1991_df.columns, 'Station No ID  (Zhang)', 'Station No ID (Zhang)', 'Station No ID'),
    'waterbody': _find_column(source_1991_df.columns, 'Water body', 'Waterbody'),
    'Year sampled': _find_column(source_1991_df.columns, 'Year sampled'),
    'Latitude': _find_column(source_1991_df.columns, 'Latitude'),
    'Longitude': _find_column(source_1991_df.columns, 'Longitude'),
}
chemical_cols_1991 = {
    'Co': _find_column(source_1991_df.columns, 'COBALT'),
    'Al': _find_column(source_1991_df.columns, 'ALUMINUM'),
    'Ni': _find_column(source_1991_df.columns, 'NICKEL'),
    'Mn': _find_column(source_1991_df.columns, 'MANGANESE'),
    'Fe': _find_column(source_1991_df.columns, 'IRON'),
    'Cr': _find_column(source_1991_df.columns, 'CHROMIUM'),
    'Cu': _find_column(source_1991_df.columns, 'COPPER'),
    'Hg': _find_column(source_1991_df.columns, 'MERCURY'),
    'Pb': _find_column(source_1991_df.columns, 'LEAD'),
    'Zn': _find_column(source_1991_df.columns, 'ZINC'),
    'SumPCBs': _find_column(source_1991_df.columns, 'PCBs'),
    'Cd': _find_column(source_1991_df.columns, 'CADMIUM'),
    'OCS': _find_column(source_1991_df.columns, 'OCTACHLOROSTYRENE'),
    "p,p'-DDE": _find_column(source_1991_df.columns, "p,p'-DDE"),
    'As': _find_column(source_1991_df.columns, 'ARSENIC'),
    'Ca': _find_column(source_1991_df.columns, 'CALCIUM'),
}

site_1991_df = source_1991_df.loc[source_1991_df[meta_cols_1991['Integrated Code']].notna()].copy()
rng_1991 = np.random.default_rng(1991)
imputation_rows = []

for chem in CHEMICAL_ORDER_1991:
    source_col = chemical_cols_1991[chem]
    values = pd.to_numeric(site_1991_df[source_col], errors='coerce')
    missing_mask = values.isna()
    missing_before = int(missing_mask.sum())
    detection_limit = pd.to_numeric(pd.Series([detection_row_1991[source_col]]), errors='coerce').iloc[0]
    imputed_count = 0

    if missing_before > 0 and pd.notna(detection_limit):
        fill_values = rng_1991.uniform(0.2 * detection_limit, 0.5 * detection_limit, size=missing_before)
        values.loc[missing_mask] = fill_values
        imputed_count = missing_before

    site_1991_df[chem] = values
    imputation_rows.append({
        'Chemical': chem,
        'Source column': source_col,
        'Detection limit': detection_limit,
        'Missing before fill': missing_before,
        'Filled with uniform[0.2X, 0.5X]': imputed_count,
        'Missing after fill': int(site_1991_df[chem].isna().sum()),
    })

cleaned_1991_df = pd.DataFrame({
    'Integrated Code': site_1991_df[meta_cols_1991['Integrated Code']],
    'Station No ID': site_1991_df[meta_cols_1991['Station No ID']],
    'waterbody': site_1991_df[meta_cols_1991['waterbody']],
    'Year sampled': site_1991_df[meta_cols_1991['Year sampled']],
    'Latitude': site_1991_df[meta_cols_1991['Latitude']],
    'Longitude': site_1991_df[meta_cols_1991['Longitude']],
    **{chem: site_1991_df[chem] for chem in CHEMICAL_ORDER_1991},
})[
    ['Integrated Code', 'Station No ID', 'waterbody', 'Year sampled', 'Latitude', 'Longitude', *CHEMICAL_ORDER_1991]
]

for col in ['Integrated Code', 'Station No ID', 'waterbody']:
    cleaned_1991_df[col] = cleaned_1991_df[col].astype('string').str.strip()
    cleaned_1991_df.loc[cleaned_1991_df[col] == '', col] = pd.NA

cleaned_1991_df['Year sampled'] = pd.to_numeric(cleaned_1991_df['Year sampled'], errors='coerce').astype('Int64')
cleaned_1991_df['Latitude'] = pd.to_numeric(cleaned_1991_df['Latitude'], errors='coerce')
cleaned_1991_df['Longitude'] = pd.to_numeric(cleaned_1991_df['Longitude'], errors='coerce')
for chem in CHEMICAL_ORDER_1991:
    cleaned_1991_df[chem] = pd.to_numeric(cleaned_1991_df[chem], errors='coerce')

units_1991_df = pd.DataFrame([
    {
        'Chemical': chem,
        'Source column': chemical_cols_1991[chem],
        'Unit from transposed sheet': units_row_1991[chemical_cols_1991[chem]],
    }
    for chem in CHEMICAL_ORDER_1991
])

imputation_1991_df = pd.DataFrame(imputation_rows)

with pd.ExcelWriter(DR_OUTPUT_PATH) as writer:
    cleaned_1991_df.to_excel(writer, sheet_name='cleaned_data', index=False)
    units_1991_df.to_excel(writer, sheet_name='units', index=False)

print(f'1991 source rows read: {len(source_1991_df)}')
print(f'1991 site rows written: {len(cleaned_1991_df)}')
print(f'1991 output workbook: {DR_OUTPUT_PATH}')
print('\nMissing-value fill summary:')
display(imputation_1991_df)
print('\nCleaned 1991 DR data preview:')
display(cleaned_1991_df.head())
print('\n1991 chemical units:')
display(units_1991_df)


1991 source rows read: 82
1991 site rows written: 77
1991 output workbook: /Users/gufeng/2025_Winter/Thesis_Project/ThesisProject/CodeBase/MetaData/FinalCleaningWork/intermiediate_datasets/cleaned_1991_DR_sites.xlsx

Missing-value fill summary:


,Chemical,Source column,Detection limit,Missing before fill,"Filled with uniform[0.2X, 0.5X]",Missing after fill
0,Co,COBALT,NaN,0,0,0
1,Al,ALUMINUM,NaN,0,0,0
2,Ni,NICKEL,NaN,0,0,0
3,Mn,MANGANESE,NaN,0,0,0
4,Fe,IRON,0.20,0,0,0
5,Cr,CHROMIUM,NaN,0,0,0
6,Cu,COPPER,NaN,0,0,0
7,Hg,MERCURY,0.01,1,1,0
8,Pb,LEAD,NaN,0,0,0
9,Zn,ZINC,NaN,0,0,0



Cleaned 1991 DR data preview:


,Integrated Code,Station No ID,waterbody,Year sampled,Latitude,Longitude,Co,Al,Ni,Mn,...,Cu,Hg,Pb,Zn,SumPCBs,Cd,OCS,"p,p'-DDE",As,Ca
1,DR-123,A,DR,1991,42.343889,-82.914444,3.1,6.1,13.0,240.0,...,8.0,0.05,4.7,35.0,7.790600,0.09,0.245885,0.205461,3.8,49.0
2,DR-124,B,DR,1991,42.339722,-82.923056,9.4,19.0,29.0,390.0,...,26.0,0.13,28.0,110.0,4.916869,1.10,0.462150,0.306288,8.9,49.0
3,DR-125,C,DR,1991,42.341111,-82.931944,2.1,3.5,23.0,140.0,...,21.0,0.02,87.0,96.0,8.312860,0.17,0.329102,0.278780,3.5,28.0
4,DR-126,D,DR,1991,42.345833,-82.946389,6.6,12.0,20.0,330.0,...,18.0,0.20,21.0,67.0,9.647125,0.74,0.226157,4.000000,4.9,36.0
5,DR-112,6FB,DR,1991,42.335556,-82.956667,5.2,13.0,23.0,310.0,...,19.0,0.16,16.0,75.0,7.546028,0.31,0.392124,0.314199,5.8,39.0



1991 chemical units:


,Chemical,Source column,Unit from transposed sheet
0,Co,COBALT,µg/g
1,Al,ALUMINUM,mg/g
2,Ni,NICKEL,µg/g
3,Mn,MANGANESE,µg/g
4,Fe,IRON,mg/g
5,Cr,CHROMIUM,µg/g
6,Cu,COPPER,µg/g
7,Hg,MERCURY,µg/g
8,Pb,LEAD,µg/g
9,Zn,ZINC,µg/g


In [13]:
print(f"The shape of the cleaned 1991 DR sites dataframe is {cleaned_1991_df.shape}")

The shape of the cleaned 1991 DR sites dataframe is (77, 22)


## Concate and deduplicate the two chemical sets

This section reads the two cleaned chemical workbooks, compares and aligns their 16 chemical units to the cleaned major-set unit array, concatenates them with a source flag, removes major-set rows that have 1991 DR counterparts, and exports the finalized unique site table plus chemical summary statistics to `finalized_datasets`.

In [8]:
FINAL_OUTPUT_DIR = DATA_DIR.parent / 'finalized_datasets'
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAJOR_WORKBOOK_PATH = OUTPUT_PATH
DR_WORKBOOK_PATH = DR_OUTPUT_PATH
CHEMICAL_ORDER_FINAL = CHEMICAL_ORDER.copy()
META_COLS_FINAL = ['Integrated Code', 'Station No ID', 'waterbody', 'Year sampled', 'Latitude', 'Longitude']
SOURCE_COL = 'row_source'

TO_UGG = {
    'mg/g': 1000.0,
    'ug/g': 1.0,
    'µg/g': 1.0,
    'μg/g': 1.0,
    'ng/g': 0.001,
}

major_data_final = pd.read_excel(MAJOR_WORKBOOK_PATH, sheet_name='cleaned_data')
major_units_final = pd.read_excel(MAJOR_WORKBOOK_PATH, sheet_name='inferred_units')
dr_data_final = pd.read_excel(DR_WORKBOOK_PATH, sheet_name='cleaned_data')
dr_units_final = pd.read_excel(DR_WORKBOOK_PATH, sheet_name='units')

major_unit_map = dict(zip(major_units_final['Chemical'], major_units_final['Inferred unit']))
dr_unit_map = dict(zip(dr_units_final['Chemical'], dr_units_final['Unit from transposed sheet']))
target_unit_map = {chem: major_unit_map[chem] for chem in CHEMICAL_ORDER_FINAL}

unit_comparison_df = pd.DataFrame({
    'Chemical': CHEMICAL_ORDER_FINAL,
    'major_set_unit': [major_unit_map[chem] for chem in CHEMICAL_ORDER_FINAL],
    '1991_DR_unit': [dr_unit_map[chem] for chem in CHEMICAL_ORDER_FINAL],
    'target_unit': [target_unit_map[chem] for chem in CHEMICAL_ORDER_FINAL],
})
unit_comparison_df['major_to_target_factor'] = [
    TO_UGG[major_unit_map[chem]] / TO_UGG[target_unit_map[chem]]
    for chem in CHEMICAL_ORDER_FINAL
]
unit_comparison_df['dr_to_target_factor'] = [
    TO_UGG[dr_unit_map[chem]] / TO_UGG[target_unit_map[chem]]
    for chem in CHEMICAL_ORDER_FINAL
]
unit_comparison_df['units_already_match'] = (
    unit_comparison_df['major_set_unit'] == unit_comparison_df['1991_DR_unit']
)

major_aligned_df = major_data_final[META_COLS_FINAL + CHEMICAL_ORDER_FINAL].copy()
dr_aligned_df = dr_data_final[META_COLS_FINAL + CHEMICAL_ORDER_FINAL].copy()

for chem in CHEMICAL_ORDER_FINAL:
    major_aligned_df[chem] = pd.to_numeric(major_aligned_df[chem], errors='coerce') * (
        TO_UGG[major_unit_map[chem]] / TO_UGG[target_unit_map[chem]]
    )
    dr_aligned_df[chem] = pd.to_numeric(dr_aligned_df[chem], errors='coerce') * (
        TO_UGG[dr_unit_map[chem]] / TO_UGG[target_unit_map[chem]]
    )

major_aligned_df[SOURCE_COL] = 'major_set'
dr_aligned_df[SOURCE_COL] = '1991_DR'


def _normalize_station_id(value):
    if pd.isna(value):
        return pd.NA
    token = ''.join(ch for ch in str(value).upper() if ch.isalnum())
    if token.endswith('FB'):
        token = token[:-2]
    if token == '20':
        token = '200'
    return token if token else pd.NA


def _normalize_integrated_code(value):
    if pd.isna(value):
        return pd.NA
    token = ''.join(ch for ch in str(value).upper() if ch.isalnum())
    return token if token else pd.NA


for frame in [major_aligned_df, dr_aligned_df]:
    frame['_station_norm'] = frame['Station No ID'].map(_normalize_station_id)
    frame['_integrated_code_norm'] = frame['Integrated Code'].map(_normalize_integrated_code)

station_keys_1991 = set(dr_aligned_df['_station_norm'].dropna())
integrated_code_keys_1991 = set(dr_aligned_df['_integrated_code_norm'].dropna())

major_aligned_df['matched_by_station'] = major_aligned_df['_station_norm'].isin(station_keys_1991)
major_aligned_df['matched_by_integrated_code'] = major_aligned_df['_integrated_code_norm'].isin(integrated_code_keys_1991)
major_aligned_df['found_in_1991_DR'] = (
    major_aligned_df['matched_by_station'] | major_aligned_df['matched_by_integrated_code']
)

major_dropped_df = major_aligned_df.loc[major_aligned_df['found_in_1991_DR']].copy()
major_kept_df = major_aligned_df.loc[~major_aligned_df['found_in_1991_DR']].copy()

complete_unique_df = pd.concat([dr_aligned_df, major_kept_df], ignore_index=True, sort=False)
complete_unique_df = complete_unique_df[META_COLS_FINAL + CHEMICAL_ORDER_FINAL + [SOURCE_COL]].copy()
complete_unique_df = complete_unique_df.sort_values(
    [SOURCE_COL, 'Integrated Code', 'Station No ID'],
    ascending=[True, True, True],
    kind='stable',
).reset_index(drop=True)

complete_row_amount = len(complete_unique_df)
final_output_path = FINAL_OUTPUT_DIR / '322_sites_chemical_data.xlsx'

chemical_summary_rows = []
for chem in CHEMICAL_ORDER_FINAL:
    series = pd.to_numeric(complete_unique_df[chem], errors='coerce')
    chemical_summary_rows.append({
        'Chemical': chem,
        'major_set_unit': major_unit_map[chem],
        '1991_DR_unit': dr_unit_map[chem],
        'target_unit': target_unit_map[chem],
        'major_to_target_factor': TO_UGG[major_unit_map[chem]] / TO_UGG[target_unit_map[chem]],
        'dr_to_target_factor': TO_UGG[dr_unit_map[chem]] / TO_UGG[target_unit_map[chem]],
        'mean': series.mean(),
        'std': series.std(ddof=1),
        'q00_min': series.quantile(0.00),
        'q25': series.quantile(0.25),
        'q50_median': series.quantile(0.50),
        'q75': series.quantile(0.75),
        'q100_max': series.quantile(1.00),
    })

chemical_summary_df = pd.DataFrame(chemical_summary_rows)

with pd.ExcelWriter(final_output_path) as writer:
    complete_unique_df.to_excel(writer, sheet_name='complete_data', index=False)
    chemical_summary_df.to_excel(writer, sheet_name='chemical_info', index=False)

all_dropped_found = bool(major_dropped_df['found_in_1991_DR'].all()) if not major_dropped_df.empty else True
major_dropped_report_df = major_dropped_df[
    ['Integrated Code', 'Station No ID', 'waterbody', 'Year sampled', 'matched_by_station', 'matched_by_integrated_code', 'found_in_1991_DR']
] .sort_values(['Integrated Code', 'Station No ID'], kind='stable').reset_index(drop=True)

print('Unit comparison between the two cleaned inputs:')
display(unit_comparison_df)

print('\nConcatenation and deduplication summary:')
print(f'  major rows before deduplication: {len(major_aligned_df)}')
print(f'  1991 DR rows: {len(dr_aligned_df)}')
print(f'  major rows dropped because a 1991 DR counterpart exists: {len(major_dropped_df)}')
print(f'  major rows kept: {len(major_kept_df)}')
print(f'  final unique row count: {complete_row_amount}')
print(f'  all dropped major rows found in 1991 DR set: {all_dropped_found}')

print('\nDropped major-set rows:')
display(major_dropped_report_df)

print('\nFinal unique complete dataset preview:')
display(complete_unique_df.head())

print('\nChemical unit + descriptive statistics table:')
display(chemical_summary_df)

print(f'\nFinalized output workbook: {final_output_path}')

Unit comparison between the two cleaned inputs:


,Chemical,major_set_unit,1991_DR_unit,target_unit,major_to_target_factor,dr_to_target_factor,units_already_match
0,Co,ug/g,µg/g,ug/g,1.0,1.0,False
1,Al,ug/g,mg/g,ug/g,1.0,1000.0,False
2,Ni,ug/g,µg/g,ug/g,1.0,1.0,False
3,Mn,ug/g,µg/g,ug/g,1.0,1.0,False
4,Fe,ug/g,mg/g,ug/g,1.0,1000.0,False
5,Cr,ug/g,µg/g,ug/g,1.0,1.0,False
6,Cu,ug/g,µg/g,ug/g,1.0,1.0,False
7,Hg,ng/g,µg/g,ng/g,1.0,1000.0,False
8,Pb,ug/g,µg/g,ug/g,1.0,1.0,False
9,Zn,ug/g,µg/g,ug/g,1.0,1.0,False



Concatenation and deduplication summary:
  major rows before deduplication: 245
  1991 DR rows: 77
  major rows dropped because a 1991 DR counterpart exists: 0
  major rows kept: 245
  final unique row count: 322
  all dropped major rows found in 1991 DR set: True

Dropped major-set rows:


,Integrated Code,Station No ID,waterbody,Year sampled,matched_by_station,matched_by_integrated_code,found_in_1991_DR



Final unique complete dataset preview:


,Integrated Code,Station No ID,waterbody,Year sampled,Latitude,Longitude,Co,Al,Ni,Mn,...,Hg,Pb,Zn,SumPCBs,Cd,OCS,"p,p'-DDE",As,Ca,row_source
0,DR-100,44FB,DR,1991.0,42.108889,-83.114722,4.4,10000.0,17.0,260.0,...,200.000000,16.0,81.0,6.769647,1.00000,0.244835,0.476767,4.6,52060.0,1991_DR
1,DR-101,45FB,DR,1991.0,42.087500,-83.113056,7.1,16000.0,23.0,340.0,...,230.000000,24.0,100.0,6.843846,1.50000,0.309684,0.249242,6.0,52230.0,1991_DR
2,DR-102,46FB,DR,1991.0,42.099722,-83.179444,5.4,4500.0,30.0,470.0,...,180.000000,44.0,190.0,8.802855,2.20000,0.352657,0.388558,6.6,42000.0,1991_DR
3,DR-103,47FB,DR,1991.0,42.055000,-83.124722,2.0,2900.0,5.6,140.0,...,4.417726,4.9,25.0,9.168158,0.02346,0.290229,0.224816,1.7,12510.0,1991_DR
4,DR-104,48FB,DR,1991.0,42.055278,-83.135556,3.6,5900.0,18.0,310.0,...,170.000000,18.0,88.0,75.000000,1.70000,2.000000,0.493464,4.2,53560.0,1991_DR



Chemical unit + descriptive statistics table:


,Chemical,major_set_unit,1991_DR_unit,target_unit,major_to_target_factor,dr_to_target_factor,mean,std,q00_min,q25,q50_median,q75,q100_max
0,Co,ug/g,µg/g,ug/g,1.0,1.0,5.946208,2.746235,1.455000,3.800000,5.400000,7.327750,14.488175
1,Al,ug/g,mg/g,ug/g,1.0,1000.0,8220.742986,6186.304201,1397.000000,3978.692913,5877.598864,10222.262557,29257.989285
2,Ni,ug/g,µg/g,ug/g,1.0,1.0,23.728659,45.968750,2.097337,7.401500,14.642189,25.905437,686.120673
3,Mn,ug/g,µg/g,ug/g,1.0,1.0,243.501069,221.939562,33.163576,126.840517,194.828007,286.200000,2100.000000
4,Fe,ug/g,mg/g,ug/g,1.0,1000.0,19178.359562,17386.739841,0.000170,9726.500000,15000.000000,22933.842906,143333.333333
5,Cr,ug/g,µg/g,ug/g,1.0,1.0,27.533250,29.961034,2.220000,10.044982,18.116777,31.941372,260.000000
6,Cu,ug/g,µg/g,ug/g,1.0,1.0,37.354158,54.831742,0.000300,12.725000,21.021924,38.290549,530.000000
7,Hg,ng/g,µg/g,ng/g,1.0,1000.0,1147.131133,1658.219094,4.417726,154.625000,509.094330,1285.019355,11700.000000
8,Pb,ug/g,µg/g,ug/g,1.0,1.0,33.228514,104.901903,0.000120,1.736301,5.617500,24.249925,1100.000000
9,Zn,ug/g,µg/g,ug/g,1.0,1.0,192.509307,290.903812,0.000457,33.764411,79.977068,273.067751,2031.402966



Finalized output workbook: /Users/gufeng/2025_Winter/Thesis_Project/ThesisProject/CodeBase/MetaData/FinalCleaningWork/finalized_datasets/322_sites_chemical_data.xlsx


## Merge the three finalized data types by StationID

This section checks the shared sample key across the finalized taxa, chemical, and environmental workbooks, derives `StationID` for the taxa table through the chemical table when needed, keeps only sites present in all three datasets, and writes the merged workbook to the top-level `FinalCleaningWork` folder with grouped column headers.

In [9]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font

FINALIZED_DATA_DIR = DATA_DIR.parent / 'finalized_datasets'
MERGED_OUTPUT_DIR = DATA_DIR.parent

TAXA_FINAL_PATH = FINALIZED_DATA_DIR / '311_taxa_data.xlsx'
CHEM_FINAL_PATH = FINALIZED_DATA_DIR / '322_sites_chemical_data.xlsx'
ENV_FINAL_PATH = FINALIZED_DATA_DIR / '323_environmental_data.xlsx'

taxa_final_df = pd.read_excel(TAXA_FINAL_PATH)
chem_final_df = pd.read_excel(CHEM_FINAL_PATH, sheet_name='complete_data')
env_final_df = pd.read_excel(ENV_FINAL_PATH)

chemical_columns_for_merge = CHEMICAL_ORDER_FINAL.copy()
taxa_value_cols = [
    col for col in taxa_final_df.columns
    if col not in ['Source', 'Integrated Code', 'Year', 'Water body', 'Latitude', 'Longitude']
]
environmental_value_cols = [
    col for col in env_final_df.columns
    if col not in ['StationID', 'Year', 'Waterbody', 'Latitude', 'Longitude']
]


def _normalize_station_merge(value):
    if pd.isna(value):
        return pd.NA
    token = ''.join(ch for ch in str(value).upper() if ch.isalnum())
    if token.endswith('FB'):
        token = token[:-2]
    if token == '20':
        token = '200'
    return token if token else pd.NA


def _prepare_common_keys(df, year_col, waterbody_col, latitude_col='Latitude', longitude_col='Longitude'):
    out = df.copy()
    out['_year_key'] = pd.to_numeric(out[year_col], errors='coerce')
    out['_waterbody_key'] = out[waterbody_col].astype('string').str.strip().str.upper()
    out['_lat_key'] = pd.to_numeric(out[latitude_col], errors='coerce').round(6)
    out['_lon_key'] = pd.to_numeric(out[longitude_col], errors='coerce').round(6)
    return out


taxa_merge_df = _prepare_common_keys(taxa_final_df, year_col='Year', waterbody_col='Water body')
chem_merge_df = _prepare_common_keys(chem_final_df, year_col='Year sampled', waterbody_col='waterbody')
env_merge_df = _prepare_common_keys(env_final_df, year_col='Year', waterbody_col='Waterbody')

chem_merge_df['StationID'] = chem_merge_df['Station No ID'].astype('string').str.strip()
env_merge_df['StationID'] = env_merge_df['StationID'].astype('string').str.strip()
chem_merge_df['_station_key'] = chem_merge_df['StationID'].map(_normalize_station_merge)
env_merge_df['_station_key'] = env_merge_df['StationID'].map(_normalize_station_merge)

chemical_station_lookup = (
    chem_merge_df[
        ['Integrated Code', 'StationID', 'row_source', '_year_key', '_waterbody_key', '_lat_key', '_lon_key']
    ]
    .drop_duplicates(['Integrated Code', '_year_key', '_waterbody_key', '_lat_key', '_lon_key'])
    .reset_index(drop=True)
)

taxa_merge_df = taxa_merge_df.merge(
    chemical_station_lookup,
    on=['Integrated Code', '_year_key', '_waterbody_key', '_lat_key', '_lon_key'],
    how='left',
)
taxa_merge_df['_station_key'] = taxa_merge_df['StationID'].map(_normalize_station_merge)
taxa_merge_df = taxa_merge_df.rename(columns={'Source': 'taxa_source'})

key_check_df = pd.DataFrame([
    {
        'dataset': 'taxa',
        'direct_stationid_column': 'no',
        'stationid_used_for_merge': 'derived from chemical data via Integrated Code + Year + Water body + Latitude + Longitude',
        'row_count': len(taxa_merge_df),
    },
    {
        'dataset': 'chemical',
        'direct_stationid_column': 'yes (Station No ID)',
        'stationid_used_for_merge': 'Station No ID',
        'row_count': len(chem_merge_df),
    },
    {
        'dataset': 'environmental',
        'direct_stationid_column': 'yes (StationID)',
        'stationid_used_for_merge': 'StationID',
        'row_count': len(env_merge_df),
    },
])

chem_for_merge_df = chem_merge_df[
    ['Integrated Code', 'StationID', 'row_source', '_station_key', '_year_key', '_waterbody_key', '_lat_key', '_lon_key', *chemical_columns_for_merge]
].copy()
env_for_merge_df = env_merge_df[
    ['StationID', '_station_key', '_year_key', '_waterbody_key', '_lat_key', '_lon_key', *environmental_value_cols]
].copy()

taxa_chem_merged_df = taxa_merge_df.merge(
    chem_for_merge_df,
    on=['Integrated Code', 'StationID', '_station_key', '_year_key', '_waterbody_key', '_lat_key', '_lon_key'],
    how='inner',
    suffixes=('', '_chemical'),
)

fully_merged_df = taxa_chem_merged_df.merge(
    env_for_merge_df,
    on=['StationID', '_station_key', '_year_key', '_waterbody_key', '_lat_key', '_lon_key'],
    how='inner',
    suffixes=('', '_environmental'),
)

fully_merged_df = fully_merged_df.sort_values(['StationID', 'Integrated Code'], kind='stable').reset_index(drop=True)
fully_recorded_site_count = fully_merged_df['_station_key'].nunique()

unmatched_taxa_df = taxa_merge_df.loc[taxa_merge_df['StationID'].isna(), ['Integrated Code', 'Year', 'Water body', 'Latitude', 'Longitude']].copy()

sample_info_df = pd.DataFrame({
    'StationID': fully_merged_df['StationID'],
    'Integrated Code': fully_merged_df['Integrated Code'],
    'Year': fully_merged_df['Year'],
    'Water body': fully_merged_df['Water body'],
    'Latitude': fully_merged_df['Latitude'],
    'Longitude': fully_merged_df['Longitude'],
    'chemical_source': fully_merged_df['row_source'],
    'taxa_source': fully_merged_df['taxa_source'],
})
chemical_block_df = fully_merged_df[chemical_columns_for_merge].copy()
environmental_block_df = fully_merged_df[environmental_value_cols].copy()
taxa_block_df = fully_merged_df[taxa_value_cols].copy()

merged_output_df = pd.concat(
    [sample_info_df, chemical_block_df, environmental_block_df, taxa_block_df],
    axis=1,
)
merged_output_df.columns = pd.MultiIndex.from_tuples(
    [('sample_info', col) for col in sample_info_df.columns]
    + [('chemical', col) for col in chemical_block_df.columns]
    + [('environemntal', col) for col in environmental_block_df.columns]
    + [('taxa', col) for col in taxa_block_df.columns]
)

merged_output_path = MERGED_OUTPUT_DIR / f'merged_{fully_recorded_site_count}_sites_taxa_chemical_environmental_data.xlsx'

wb = Workbook()
ws = wb.active
ws.title = 'merged_data'

header_level_1 = [col[0] for col in merged_output_df.columns]
header_level_2 = [col[1] for col in merged_output_df.columns]

for col_idx, value in enumerate(header_level_1, start=1):
    ws.cell(row=1, column=col_idx, value=value)
for col_idx, value in enumerate(header_level_2, start=1):
    ws.cell(row=2, column=col_idx, value=value)

start_col = 1
while start_col <= len(header_level_1):
    end_col = start_col
    while end_col <= len(header_level_1) and header_level_1[end_col - 1] == header_level_1[start_col - 1]:
        end_col += 1
    end_col -= 1
    if end_col > start_col:
        ws.merge_cells(start_row=1, start_column=start_col, end_row=1, end_column=end_col)
    start_col = end_col + 1

for row_idx, row_values in enumerate(merged_output_df.itertuples(index=False, name=None), start=3):
    for col_idx, value in enumerate(row_values, start=1):
        ws.cell(row=row_idx, column=col_idx, value=value)

for row in [1, 2]:
    for col_idx in range(1, len(header_level_1) + 1):
        cell = ws.cell(row=row, column=col_idx)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center', vertical='center')

ws.freeze_panes = 'A3'

summary_ws = wb.create_sheet('merge_summary')
summary_ws.append(['metric', 'value'])
summary_ws.append(['taxa_rows', len(taxa_final_df)])
summary_ws.append(['chemical_rows', len(chem_final_df)])
summary_ws.append(['environmental_rows', len(env_final_df)])
summary_ws.append(['taxa_rows_with_derived_stationid', int(taxa_merge_df['StationID'].notna().sum())])
summary_ws.append(['fully_recorded_sites_across_all_3_sets', fully_recorded_site_count])
summary_ws.append(['unmatched_taxa_rows', len(unmatched_taxa_df)])
summary_ws.append([])
summary_ws.append(['key_check'])
for row in key_check_df.itertuples(index=False, name=None):
    summary_ws.append(list(row))
summary_ws.append([])
summary_ws.append(['unmatched_taxa_rows'])
summary_ws.append(list(unmatched_taxa_df.columns))
for row in unmatched_taxa_df.itertuples(index=False, name=None):
    summary_ws.append(list(row))

wb.save(merged_output_path)

print('Common-key check:')
display(key_check_df)
print(f'\nFully recorded sites across all three datasets: {fully_recorded_site_count}')
print(f'Taxa rows matched to a StationID through the chemical table: {int(taxa_merge_df["StationID"].notna().sum())} / {len(taxa_merge_df)}')
print(f'Unmatched taxa rows: {len(unmatched_taxa_df)}')
print('\nUnmatched taxa rows:')
display(unmatched_taxa_df)
print('\nMerged data preview:')
display(merged_output_df.head())
print(f'\nMerged output workbook: {merged_output_path}')


Common-key check:


,dataset,direct_stationid_column,stationid_used_for_merge,row_count
0,taxa,no,derived from chemical data via Integrated Code...,311
1,chemical,yes (Station No ID),Station No ID,322
2,environmental,yes (StationID),StationID,323



Fully recorded sites across all three datasets: 310
Taxa rows matched to a StationID through the chemical table: 310 / 311
Unmatched taxa rows: 1

Unmatched taxa rows:


,Integrated Code,Year,Water body,Latitude,Longitude
113,DR-183,1999,DR,42.259607,-83.123022



Merged data preview:


sample_info                                                         \
    StationID Integrated Code  Year Water body   Latitude  Longitude   
0      003ABC           DR-02  1999         DR  42.353685 -82.944344   
1      004ABC           DR-03  1999         DR  42.350308 -82.934247   
2      005ABC           DR-04  1999         DR  42.342434 -82.945587   
3      007ABC           DR-06  1999         DR  42.344791 -82.936142   
4        008A           DR-07  1999         DR  42.350944 -82.922672   

                                chemical                ...           taxa  \
  chemical_source taxa_source         Co            Al  ... Hydropsychidae   
0       major_set       log21   3.653624   4742.524828  ...       2.438439   
1       major_set       log21   3.164832   3509.922179  ...       0.000000   
2       major_set       log21   2.984709   3670.729827  ...       0.000000   
3       major_set       log21  10.625940  20375.805889  ...       0.000000   
4       major_set       log21   3.549781   3827.601114  ...       0.128438   

                                                                      \
  Other Trichoptera Amphipoda Dreissena     Acari Hydrozoa Hirudinea   
0               0.0  4.529993  6.082918  0.000000  0.00000   0.00000   
1               0.0  0.000000  0.801982  0.801982  0.00000   0.00000   
2               0.0  0.000000  0.172158  1.010022  0.00000   0.00000   
3               0.0  1.840242  6.475708  0.000000  1.82093   0.23027   
4               0.0  4.224262  6.266368  0.000000  0.00000   0.00000   

                                      
  Turbellaria Gastropoda Sphaeriidae  
0         0.0   0.000000         0.0  
1         0.0   0.456015         0.0  
2         0.0   0.464907         0.0  
3         0.0   0.000000         0.0  
4         0.0   0.246371         0.0  

[5 rows x 46 columns]


Merged output workbook: /Users/gufeng/2025_Winter/Thesis_Project/ThesisProject/CodeBase/MetaData/FinalCleaningWork/merged_310_sites_taxa_chemical_environmental_data.xlsx
